In [1]:
import numpy as np

In [2]:
# 约定: (B, T, E, H) 分别表示 批次/序列长度/输入维度/隐藏维度
B, E, H = 1, 128, 3

def prepare_inputs():
    """
    使用 NumPy 准备输入数据
    使用示例句子: "播放 周杰伦 的 《稻香》"
    构造最小词表和随机(可复现)词向量, 生成形状为 (B, T, E) 的输入张量。
    """
    np.random.seed(42)
    vocab = {"播放": 0, "周杰伦": 1, "的": 2, "《稻香》": 3}
    tokens = ["播放", "周杰伦", "的", "《稻香》"]
    ids = [vocab[t] for t in tokens]

    # 词向量表: (V, E)
    V = len(vocab)
    emb_table = np.random.randn(V, E).astype(np.float32)

    # 取出序列词向量并加上 batch 维度: (B, T, E)
    x_np = emb_table[ids][None]
    return tokens, x_np


In [3]:
class SimpleRNN:
    """使用 NumPy 实现的简单 RNN"""

    def __init__(self, input_size, hidden_size):
        """
        初始化 RNN 参数

        Args:
            input_size: 输入维度 E
            hidden_size: 隐藏状态维度 H
        """
        self.input_size = input_size
        self.hidden_size = hidden_size

        # 初始化权重和偏置
        self.U = np.random.randn(hidden_size, input_size) * 0.01   # 输入权重
        self.W = np.random.randn(hidden_size, hidden_size) * 0.01  # 循环权重
        self.b = np.zeros(hidden_size)                              # 偏置

    def forward(self, x, h_prev=None):
        """
        RNN 前向传播

        Args:
            x: 输入序列 (B, T, E)
            h_prev: 初始隐藏状态 (B, H)，默认为零向量

        Returns:
            h_all: 所有时间步的隐藏状态 (B, T, H)
            h_last: 最后一个时间步的隐藏状态 (B, H)
        """
        B, T, E = x.shape
        H = self.hidden_size

        # 初始化隐藏状态
        if h_prev is None:
            h_prev = np.zeros((B, H))

        # 存储所有时间步的隐藏状态
        h_all = []
        h_t = h_prev

        for t in range(T):
            # 当前时间步的输入
            x_t = x[:, t, :]  # (B, E)

            # RNN 核心公式: h_t = tanh(U·x_t + W·h_{t-1} + b)
            u = x_t @ self.U.T  # (B, H)  注意：@ 是矩阵乘法
            w = h_t @ self.W.T  # (B, H)
            h_t = np.tanh(u + w + self.b)

            h_all.append(h_t)

        # 堆叠所有时间步: (B, T, H)
        h_all = np.stack(h_all, axis=1)

        return h_all, h_t




In [4]:
# ==================== 测试代码 ====================

if __name__ == "__main__":
    print("="*60)
    print("RNN 前向传播演示")
    print("="*60)

    # 1. 准备输入
    tokens, x = prepare_inputs()
    B, T, E = x.shape
    print(f"\n【输入数据】")
    print(f"  句子: {' → '.join(tokens)}")
    print(f"  输入形状: {x.shape} (B={B}, T={T}, E={E})")
    print(f"  输入向量（前5维）:\n{x[0, 0, :5]}...")

    # 2. 创建 RNN 模型
    rnn = SimpleRNN(input_size=E, hidden_size=H)
    print(f"\n【RNN 模型】")
    print(f"  输入维度: {E}")
    print(f"  隐藏维度: {H}")
    print(f"  参数量: {E*H + H*H + H:,}")

    # 3. 前向传播
    h_all, h_last = rnn.forward(x)
    print(f"\n【前向传播结果】")
    print(f"  所有时间步隐藏状态形状: {h_all.shape} (B={B}, T={T}, H={H})")
    print(f"  最后隐藏状态形状: {h_last.shape}")

    # 4. 打印每个时间步的隐藏状态
    print(f"\n【每个时间步的隐藏状态】")
    for t, token in enumerate(tokens):
        print(f"  t={t}: '{token}' → h_{t} = {h_all[0, t]}")

    # 5. 验证信息传递
    print(f"\n【信息传递验证】")
    print(f"  h0 只包含 '{tokens[0]}' 的信息")
    print(f"  h1 包含 '{tokens[0]}' 和 '{tokens[1]}' 的信息")
    print(f"  h2 包含 '{tokens[0]}', '{tokens[1]}', '{tokens[2]}' 的信息")
    print(f"  h3 包含全部 '{' → '.join(tokens)}' 的信息")

    # 6. 验证不同顺序的结果不同
    print(f"\n【验证顺序敏感性】")

    # 创建不同顺序的输入
    tokens_rev = ["《稻香》", "的", "周杰伦", "播放"]
    ids_rev = [0, 2, 1, 3]  # 重新映射
    x_rev = x[0, [3, 2, 1, 0], :][None]  # 反转顺序

    _, h_last_rev = rnn.forward(x_rev)

    print(f"  原始顺序: {' → '.join(tokens)}")
    print(f"  反转顺序: {' → '.join(tokens_rev)}")
    print(f"  原始顺序的最后隐藏状态: {h_last[0, :5]}...")
    print(f"  反转顺序的最后隐藏状态: {h_last_rev[0, :5]}...")

    if not np.allclose(h_last, h_last_rev):
        print(f"  ✅ 结果不同！RNN 能够区分词序！")
    else:
        print(f"  ❌ 结果相同（不应该发生）")

RNN 前向传播演示

【输入数据】
  句子: 播放 → 周杰伦 → 的 → 《稻香》
  输入形状: (1, 4, 128) (B=1, T=4, E=128)
  输入向量（前5维）:
[ 0.49671414 -0.1382643   0.64768857  1.5230298  -0.23415338]...

【RNN 模型】
  输入维度: 128
  隐藏维度: 3
  参数量: 396

【前向传播结果】
  所有时间步隐藏状态形状: (1, 4, 3) (B=1, T=4, H=3)
  最后隐藏状态形状: (1, 3)

【每个时间步的隐藏状态】
  t=0: '播放' → h_0 = [ 0.22554555 -0.06922397 -0.03296116]
  t=1: '周杰伦' → h_1 = [ 0.05726691 -0.09702182 -0.08463605]
  t=2: '的' → h_2 = [ 0.03090935  0.04695418 -0.01573247]
  t=3: '《稻香》' → h_3 = [ 0.12671037  0.06709692 -0.12362833]

【信息传递验证】
  h0 只包含 '播放' 的信息
  h1 包含 '播放' 和 '周杰伦' 的信息
  h2 包含 '播放', '周杰伦', '的' 的信息
  h3 包含全部 '播放 → 周杰伦 → 的 → 《稻香》' 的信息

【验证顺序敏感性】
  原始顺序: 播放 → 周杰伦 → 的 → 《稻香》
  反转顺序: 《稻香》 → 的 → 周杰伦 → 播放
  原始顺序的最后隐藏状态: [ 0.12671037  0.06709692 -0.12362833]...
  反转顺序的最后隐藏状态: [ 0.22436832 -0.06971876 -0.03431763]...
  ✅ 结果不同！RNN 能够区分词序！
